In [1]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
# 프로젝트 루트 기준 상대경로 (notebooks/개인의것/ → ../../)
ROOT = Path('../../').resolve()
DATA_DIR = ROOT / 'data'

print(f'프로젝트 루트: {ROOT}')
print(f'데이터 경로:   {DATA_DIR}')

# 원본 데이터 로딩
orders       = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
customers    = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
order_items  = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
payments     = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
reviews      = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
products     = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers      = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
category_tr  = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')
geolocation  = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')

print('\n데이터 로딩 완료')
for name, df in [('orders', orders), ('customers', customers), ('order_items', order_items),
                 ('payments', payments), ('reviews', reviews), ('products', products),
                 ('sellers', sellers), ('category_tr', category_tr)]:
    print(f'  {name:15s}: {df.shape}')

프로젝트 루트: C:\team-oldest-olist-analysis
데이터 경로:   C:\team-oldest-olist-analysis\data

데이터 로딩 완료
  orders         : (99441, 8)
  customers      : (99441, 5)
  order_items    : (112650, 7)
  payments       : (103886, 5)
  reviews        : (99224, 7)
  products       : (32951, 9)
  sellers        : (3095, 4)
  category_tr    : (71, 2)


## 1. 타임스탬프 변환

In [3]:
# orders 타임스탬프 컬럼 datetime 변환
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col])

# reviews 날짜 컬럼 변환
reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

# order_items shipping_limit_date 변환
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

print('타임스탬프 변환 완료')
orders[timestamp_cols].dtypes

타임스탬프 변환 완료


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 2. 데이터 병합 (Master DataFrame 구성)

`customer_unique_id`를 기준으로 재구매 추적이 가능하도록 병합합니다.  
`customer_id`는 주문마다 새로 발급되므로 유저 단위 분석에는 `customer_unique_id`를 사용합니다.

In [4]:
# 제품 카테고리 영문명 합치기
products = products.merge(category_tr, on='product_category_name', how='left')

# order_items에 상품 정보 추가
items_enriched = order_items.merge(
    products[['product_id', 'product_category_name', 'product_category_name_english']],
    on='product_id', how='left'
)

# 주문별 결제 집계 (한 주문에 복수 결제수단 존재 가능)
payments_agg = (
    payments
    .groupby('order_id')
    .agg(
        total_payment_value=('payment_value', 'sum'),
        payment_type=('payment_type', lambda x: x.mode()[0]),
        payment_installments=('payment_installments', 'max')
    )
    .reset_index()
)

# 주문별 아이템 집계 (다중 아이템 주문 처리)
items_agg = (
    items_enriched
    .groupby('order_id')
    .agg(
        item_count=('order_item_id', 'max'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        product_category=('product_category_name_english',
                          lambda x: x.mode()[0] if x.notna().any() else np.nan)
    )
    .reset_index()
)

# 리뷰: order_id별 최신 1건 (중복 리뷰 제거)
reviews_dedup = (
    reviews
    .sort_values('review_answer_timestamp', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'review_score', 'review_creation_date']]
)

# 마스터 병합
df = (
    orders
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='left')
    .merge(payments_agg,  on='order_id', how='left')
    .merge(items_agg,     on='order_id', how='left')
    .merge(reviews_dedup, on='order_id', how='left')
)

print(f'마스터 DataFrame shape: {df.shape}')
df.head(3)

마스터 DataFrame shape: (99441, 19)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,total_payment_value,payment_type,payment_installments,item_count,total_price,total_freight,product_category,review_score,review_creation_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,SP,38.71,voucher,1.00,1.00,29.99,8.72,housewares,4.00,2017-10-11
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,BA,141.46,boleto,1.00,1.00,118.70,22.76,perfumery,4.00,2018-08-08
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,GO,179.12,credit_card,3.00,1.00,159.90,19.22,auto,5.00,2018-08-18


## 3. 파생 변수 생성 (Feature Engineering)

In [5]:
# --- 배송 관련 파생 변수 ---

# 배송 지연 여부 (실제 배송일 > 예상 배송일)
df['is_delayed'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype('Int8')  # nullable integer (NaN 허용)

# 지연 일수 (양수 = 지연, 음수 = 조기 배송)
df['delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

# --- 퍼널 구간별 소요 시간 ---

# 구간 1: 주문 → 결제 승인 (시간 단위)
df['time_purchase_to_approved'] = (
    df['order_approved_at'] - df['order_purchase_timestamp']
).dt.total_seconds() / 3600

# 구간 2: 결제 승인 → 물류사 인도 (판매자 처리 시간, 일 단위)
df['time_approved_to_carrier'] = (
    df['order_delivered_carrier_date'] - df['order_approved_at']
).dt.days

# 구간 3: 물류사 인도 → 고객 수령 (배송 소요 시간, 일 단위)
df['time_carrier_to_customer'] = (
    df['order_delivered_customer_date'] - df['order_delivered_carrier_date']
).dt.days

# 구간 4: 주문 → 고객 수령 (전체 리드타임, 일 단위)
df['total_lead_time_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# --- 주문 시간 특성 ---
df['purchase_year']       = df['order_purchase_timestamp'].dt.year
df['purchase_month']      = df['order_purchase_timestamp'].dt.month
df['purchase_yearmonth']  = df['order_purchase_timestamp'].dt.to_period('M')
df['purchase_dayofweek']  = df['order_purchase_timestamp'].dt.dayofweek  # 0=월요일
df['purchase_hour']       = df['order_purchase_timestamp'].dt.hour

print('파생 변수 생성 완료')
derived_cols = ['is_delayed', 'delay_days', 'time_purchase_to_approved',
                'time_approved_to_carrier', 'time_carrier_to_customer', 'total_lead_time_days']
df[derived_cols].describe()

파생 변수 생성 완료


,is_delayed,delay_days,time_purchase_to_approved,time_approved_to_carrier,time_carrier_to_customer,total_lead_time_days
count,99441.00,96476.00,99281.00,97644.00,96475.00,96476.00
mean,0.08,-11.88,10.42,2.30,8.88,12.09
std,0.27,10.18,26.04,3.56,8.75,9.55
min,0.00,-147.00,0.00,-172.00,-17.00,0.00
25%,0.00,-17.00,0.21,0.00,4.00,6.00
50%,0.00,-12.00,0.34,1.00,7.00,10.00
75%,0.00,-7.00,14.58,3.00,12.00,15.00
max,1.00,188.00,4509.18,125.00,205.00,209.00


## 4. 결측치 처리 및 이상값 필터링

In [6]:
# 전체 결측치 현황
print('=== 전체 결측치 현황 ===')
null_summary = df.isnull().sum()
print(null_summary[null_summary > 0].to_string())
print(f'\n전체 행 수: {len(df):,}')

# 주문 상태 분포 확인
print('\n=== order_status 분포 ===')
print(df['order_status'].value_counts())

=== 전체 결측치 현황 ===
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
total_payment_value                 1
payment_type                        1
payment_installments                1
item_count                        775
total_price                       775
total_freight                     775
product_category                 2185
review_score                      768
review_creation_date              768
delay_days                       2965
time_purchase_to_approved         160
time_approved_to_carrier         1797
time_carrier_to_customer         2966
total_lead_time_days             2965

전체 행 수: 99,441

=== order_status 분포 ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [7]:
# 분석용 데이터셋: 'delivered' 상태만 유지 (배송 완료된 주문만)
df_delivered = df[df['order_status'] == 'delivered'].copy()
print(f'배송 완료 주문 수: {len(df_delivered):,} ({len(df_delivered)/len(df)*100:.1f}%)')

# 이상값 제거: 음수 리드타임 (데이터 오류)
negative_mask = (
    (df_delivered['total_lead_time_days'] < 0) |
    (df_delivered['time_approved_to_carrier'] < 0) |
    (df_delivered['time_carrier_to_customer'] < 0)
)
print(f'음수 리드타임 이상값: {negative_mask.sum()}건 제거')
df_delivered = df_delivered[~negative_mask]

# review_score 결측 → 리뷰 미작성 (NaN 유지, 분석 시 제외)
print(f'\nreview_score 결측 (리뷰 미작성): {df_delivered["review_score"].isna().sum():,}건')

print(f'\n최종 분석용 DataFrame shape: {df_delivered.shape}')

배송 완료 주문 수: 96,478 (97.0%)
음수 리드타임 이상값: 1373건 제거

review_score 결측 (리뷰 미작성): 639건

최종 분석용 DataFrame shape: (95105, 30)


## 5. 전처리 결과 저장

In [9]:
# period 타입은 CSV 저장 불가 → 문자열로 변환
df_delivered['purchase_yearmonth'] = df_delivered['purchase_yearmonth'].astype(str)

# 전체 주문 포함 버전 (퍼널 분석용)
df_all_save = df.copy()
df_all_save['purchase_yearmonth'] = df_all_save['purchase_yearmonth'].astype(str)

# 저장 경로도 상대경로 사용
df_delivered.to_csv(DATA_DIR / 'olist_master_delivered.csv', index=False)
df_all_save.to_csv(DATA_DIR / 'olist_master_all.csv',        index=False)

print('저장 완료')
print(f'  olist_master_delivered.csv : {df_delivered.shape}')
print(f'  olist_master_all.csv       : {df_all_save.shape}')
print('\n컬럼 목록:')
print(df_delivered.columns.tolist())

저장 완료
  olist_master_delivered.csv : (95105, 30)
  olist_master_all.csv       : (99441, 30)

컬럼 목록:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_state', 'total_payment_value', 'payment_type', 'payment_installments', 'item_count', 'total_price', 'total_freight', 'product_category', 'review_score', 'review_creation_date', 'is_delayed', 'delay_days', 'time_purchase_to_approved', 'time_approved_to_carrier', 'time_carrier_to_customer', 'total_lead_time_days', 'purchase_year', 'purchase_month', 'purchase_yearmonth', 'purchase_dayofweek', 'purchase_hour']


In [ ]:
df_delivered["is_delayed"].value_counts()

is_delayed
0    87312
1     7793
Name: count, dtype: Int64

: 